In [0]:
%pip install pdfplumber

In [0]:
import pdfplumber
import os
import uuid
import hashlib
from datetime import datetime
from pyspark.sql import Row
from pyspark.sql.functions import col

In [0]:
RAW_ROOT_PATH = "/Volumes/workspace/legal_data/raw_documents/legal_datasets/"
BRONZE_PATH = "/Volumes/workspace/legal_data/bronze/legal_documents/"

In [0]:
def discover_pdf_files(root_dir):
    pdf_files = []

    for category in os.listdir(root_dir):
        category_path = os.path.join(root_dir, category)

        if os.path.isdir(category_path):
            for file in os.listdir(category_path):
                if file.lower().endswith(".pdf"):
                    full_path = os.path.join(category_path, file)
                    pdf_files.append((category, file, full_path))

    return pdf_files

In [0]:
pdf_files = discover_pdf_files(RAW_ROOT_PATH)

print(f"Total PDFs discovered: {len(pdf_files)}")
pdf_files[:5]

In [0]:
def generate_file_hash(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

In [0]:
def detect_scanned_pdf(text):
    return True if len(text.strip()) < 50 else False

In [0]:
def extract_text_from_pdf(path):
    text_content = []
    page_count = 0

    with pdfplumber.open(path) as pdf:
        page_count = len(pdf.pages)

        for page in pdf.pages:
            page_text = page.extract_text() or ""
            text_content.append(page_text)

    return "\n".join(text_content), page_count

In [0]:
existing_hashes = set()

if os.path.exists("/dbfs" + BRONZE_PATH):
    try:
        existing_df = spark.read.format("delta").load(BRONZE_PATH)
        existing_hashes = set(row.file_hash for row in existing_df.select("file_hash").distinct().collect())
        print(f"Loaded {len(existing_hashes)} existing hashes")
    except:
        print("No existing bronze table found")

In [0]:
records = []

for category, file_name, path in pdf_files:

    try:
        file_hash = generate_file_hash(path)

        # Skip duplicates
        if file_hash in existing_hashes:
            continue

        text, page_count = extract_text_from_pdf(path)

        file_size_kb = os.path.getsize(path) / 1024
        modified_time = datetime.fromtimestamp(os.path.getmtime(path))
        is_scanned = detect_scanned_pdf(text)

        records.append(Row(
            doc_id=str(uuid.uuid4()),
            file_name=file_name,
            category=category,
            file_path=path,
            raw_text=text,
            page_count=page_count,
            file_size_kb=file_size_kb,
            file_hash=file_hash,
            modified_time=modified_time,
            is_scanned=is_scanned,
            ingestion_time=datetime.now(),
            status="success"
        ))

    except Exception as e:
        records.append(Row(
            doc_id=str(uuid.uuid4()),
            file_name=file_name,
            category=category,
            file_path=path,
            raw_text="",
            page_count=0,
            file_size_kb=0,
            file_hash="",
            modified_time=None,
            is_scanned=None,
            ingestion_time=datetime.now(),
            status=f"failed: {str(e)}"
        ))

In [0]:
print(len(records))

In [0]:
bronze_df = spark.createDataFrame(records)

bronze_df.display()

In [0]:
bronze_df.write.format("delta") \
    .mode("append") \
    .save(BRONZE_PATH)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.default.bronze_legal_documents
USING DELTA;

In [0]:
bronze_df = bronze_df.withColumn("ingestion_time", current_timestamp())

bronze_df.write \
    .format("delta") \
    .saveAsTable("workspace.default.bronze_legal_documents")

In [0]:
%sql
SELECT COUNT(*) FROM workspace.default.bronze_legal_documents;


In [0]:
%sql
select * from workspace.default.bronze_legal_documents
limit 10

In [0]:
%sql
SELECT doc_id, file_name, category
FROM workspace.default.bronze_legal_documents
LIMIT 61;

In [0]:
df = spark.read.format("delta").load(BRONZE_PATH)

print("Total records:", df.count())

df.select("category").distinct().display()

df.filter(col("status") != "success").display()